# Where does TSBench-Forge live in benchmark space?

**Question.** Do our live-catalog series occupy the same region of time-series
"character space" as the established TSFM benchmarks — [GIFT-Eval](https://hf.co/datasets/Salesforce/GiftEval)
(arXiv:2410.10393), [TIME](https://hf.co/datasets/Real-TSF/TIME) (arXiv:2602.12147) and
[BOOM](https://hf.co/datasets/Datadog/BOOM) (arXiv:2505.14766) — or a different one?

**Method** (the analysis the BOOM paper uses to show observability data is *unlike*
GIFT-Eval): sample series from each benchmark, describe every series with a fixed
vector of structural features (trend, seasonality, spectral, distributional,
intermittency — in the spirit of `tsfeatures`/catch22), robust-standardise, project
with PCA, and compare the occupied regions.

**Reading it:** overlap = our benchmark tests the same regimes with fresher,
uncontaminated data; distinct mass = regimes the incumbents undertest; a *wider*
spread = broader DGP coverage per series sampled.


In [1]:
import os, sys, warnings
from pathlib import Path
import numpy as np

warnings.filterwarnings("ignore")
_cands = [Path.cwd(), Path.cwd() / "src", Path.cwd().parent / "src",
          Path(os.environ.get("TSBENCH_SRC", "/root/TSBench-Forge/src"))]
SRC = next(p for p in _cands if (p / "scraped_source.py").exists())
os.chdir(SRC); sys.path.insert(0, str(SRC))

SEED = 20260704
N_PER_BENCH = 300          # series sampled per benchmark
WIN_MAX, WIN_MIN = 1024, 96  # analysis window: last WIN_MAX points, need >= WIN_MIN
rng = np.random.default_rng(SEED)
print("src:", SRC)

src: /root/TSBench-Forge/src


## 1. Structural feature vector

Twelve features per series, computed on the (z-scored) last `WIN_MAX` observations.
Chosen to span the axes the BOOM paper highlights: trend, seasonal structure,
spectral character, tail/asymmetry, local stability, and the zero-inflation /
flat-spell behaviour typical of operational data.

In [2]:
def _acf(x, lag):
    x = x - x.mean()
    d = float(np.dot(x, x))
    return float(np.dot(x[:-lag], x[lag:]) / d) if d > 0 and lag < len(x) else 0.0

def features(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < WIN_MIN:
        return None
    x = x[-WIN_MAX:]
    n = len(x)
    sd = x.std() or 1.0
    z = (x - x.mean()) / sd
    t = np.arange(n)
    # trend strength: R^2 of a linear fit
    b, a = np.polyfit(t, z, 1)
    resid = z - (a + b * t)
    trend_r2 = 1.0 - resid.var() / max(z.var(), 1e-12)
    # seasonality: strongest ACF over candidate lags on the detrended series
    lags = [l for l in (4, 7, 12, 24, 48, 96, 168, 288) if l < n // 2]
    seas = max((abs(_acf(resid, l)) for l in lags), default=0.0)
    # spectral entropy + flatness on the detrended, windowed series
    y = resid * np.hanning(n)
    psd = np.abs(np.fft.rfft(y))[1:] ** 2 + 1e-20
    p = psd / psd.sum()
    spec_entropy = float(-(p * np.log(p)).sum() / np.log(len(p)))
    spec_flat = float(np.exp(np.mean(np.log(psd))) / psd.mean())
    # local stability: variance of tile means / tile variances (stability, lumpiness)
    k = max(n // 10, 2)
    tiles = [z[i:i + k] for i in range(0, n - k + 1, k)]
    stability = float(np.var([tt.mean() for tt in tiles]))
    lumpiness = float(np.var([tt.var() for tt in tiles]))
    diffs = np.diff(x)
    # longest flat spell + zero fraction (intermittency / operational character)
    flat = np.max(np.diff(np.flatnonzero(np.concatenate(([True], diffs != 0, [True]))))) / n
    zero_frac = float(np.mean(x == 0))
    from scipy import stats as st
    return dict(trend_r2=float(np.clip(trend_r2, 0, 1)), seasonality=seas,
                spec_entropy=spec_entropy, spec_flatness=spec_flat,
                acf1=_acf(z, 1), acf10=_acf(z, 10) if n > 20 else 0.0,
                skew=float(np.clip(st.skew(x), -10, 10)),
                kurtosis=float(np.clip(st.kurtosis(x), -10, 50)),
                stability=float(np.clip(stability, 0, 10)),
                lumpiness=float(np.clip(lumpiness, 0, 50)),
                flat_spell=float(flat), zero_frac=zero_frac)

FEATS = list(features(np.random.default_rng(0).normal(size=256)).keys())
print(len(FEATS), "features:", FEATS)

12 features: ['trend_r2', 'seasonality', 'spec_entropy', 'spec_flatness', 'acf1', 'acf10', 'skew', 'kurtosis', 'stability', 'lumpiness', 'flat_spell', 'zero_frac']


## 2. Sample series from the four benchmarks

- **TSBench-Forge**: windows served exactly as the benchmark serves them
  (`ScrapedLiveSource`, breadth-balanced, chronological, gap-aware).
- **GIFT-Eval**: `Salesforce/GiftEvalParquet` (`history_value` per series).
- **TIME**: `Real-TSF/TIME` arrow shards (one per dataset × frequency; multivariate
  targets contribute each variate).
- **BOOM**: `Datadog/BOOM` arrow shards (random subset of the 2,807 dataset dirs).

Sampling is seeded; every download is a single small file via `hf_hub_download`.

In [3]:
from huggingface_hub import HfApi, hf_hub_download
import datasets as hfds
import pandas as pd

api = HfApi()
samples = {}   # bench -> list of 1-D arrays

# --- ours: exactly what the benchmark serves
from scraped_source import ScrapedLiveSource
src = ScrapedLiveSource("sources/sources.yaml", "sources/data", min_series_length=WIN_MIN + 32)
metas = src.pull_meta(N_PER_BENCH, 384, np.random.default_rng(SEED))
samples["TSBench-Forge"] = [m.motif for m in metas]
print("ours:", len(samples["TSBench-Forge"]))

ours: 300


In [4]:
def _take(bench, arrs):
    out = samples.setdefault(bench, [])
    for a in arrs:
        if len(out) >= N_PER_BENCH: break
        a = np.asarray(a, dtype=float)
        if np.isfinite(a).sum() >= WIN_MIN: out.append(a)

# --- GIFT-Eval
gift_files = sorted(f for f in api.list_repo_files("Salesforce/GiftEvalParquet", repo_type="dataset")
                    if f.endswith("train.parquet"))
rng_g = np.random.default_rng(SEED)
for f in rng_g.permutation(gift_files):
    if len(samples.get("GIFT-Eval", [])) >= N_PER_BENCH: break
    df = pd.read_parquet(hf_hub_download("Salesforce/GiftEvalParquet", f, repo_type="dataset"))
    rows = df.sample(min(8, len(df)), random_state=SEED)
    _take("GIFT-Eval", [np.asarray(v, dtype=float) for v in rows["history_value"]])
print("GIFT-Eval:", len(samples["GIFT-Eval"]))

GIFT-Eval: 300


In [5]:
# --- TIME (each shard = one dataset; multivariate target contributes each variate)
time_files = sorted(f for f in api.list_repo_files("Real-TSF/TIME", repo_type="dataset")
                    if f.endswith(".arrow"))
for f in np.random.default_rng(SEED).permutation(time_files):
    if len(samples.get("TIME", [])) >= N_PER_BENCH: break
    d = hfds.Dataset.from_file(hf_hub_download("Real-TSF/TIME", f, repo_type="dataset"))
    arrs = []
    for row in d:
        tgt = row["target"]
        arrs.extend(tgt if isinstance(tgt[0], (list, np.ndarray)) else [tgt])
    _take("TIME", arrs[:12])
print("TIME:", len(samples["TIME"]))

TIME: 300


In [6]:
# --- BOOM (random subset of dataset dirs)
boom_files = sorted(f for f in api.list_repo_files("Datadog/BOOM", repo_type="dataset")
                    if f.endswith(".arrow"))
for f in np.random.default_rng(SEED).permutation(boom_files):
    if len(samples.get("BOOM", [])) >= N_PER_BENCH: break
    try:
        d = hfds.Dataset.from_file(hf_hub_download("Datadog/BOOM", f, repo_type="dataset"))
    except Exception:
        continue
    arrs = []
    for row in d.select(range(min(3, len(d)))):
        tgt = row["target"]
        arrs.extend(tgt if isinstance(tgt[0], (list, np.ndarray)) else [tgt])
    _take("BOOM", arrs)
print("BOOM:", len(samples["BOOM"]))

BOOM: 300


## 3. Feature matrix → robust standardisation → PCA

In [7]:
rows, labels = [], []
for bench, arrs in samples.items():
    for a in arrs:
        f = features(a)
        if f is not None:
            rows.append([f[k] for k in FEATS]); labels.append(bench)
X = np.array(rows); labels = np.array(labels)
finite = np.isfinite(X).all(axis=1)
if (~finite).any():
    print(f"dropping {int((~finite).sum())} series with non-finite features (degenerate/constant)")
    X, labels = X[finite], labels[finite]
print("feature matrix:", X.shape, "| per bench:", {b: int((labels == b).sum()) for b in dict.fromkeys(labels)})

med = np.median(X, axis=0)
iqr = np.subtract(*np.percentile(X, [75, 25], axis=0)); iqr[iqr == 0] = 1.0
Xz = np.clip((X - med) / iqr, -5, 5)

from sklearn.decomposition import PCA
pca = PCA(n_components=4, random_state=SEED)
P = pca.fit_transform(Xz)
print("explained variance:", np.round(pca.explained_variance_ratio_, 3),
      "| PC1+PC2 =", round(pca.explained_variance_ratio_[:2].sum(), 3))
load = pd.DataFrame(pca.components_[:2].T, index=FEATS, columns=["PC1", "PC2"])
print("\ntop loadings:\n", load.reindex(load.abs().max(axis=1).sort_values(ascending=False).index).round(2).head(8))

dropping 18 series with non-finite features (degenerate/constant)
feature matrix: (1182, 12) | per bench: {np.str_('TSBench-Forge'): 287, np.str_('GIFT-Eval'): 297, np.str_('TIME'): 300, np.str_('BOOM'): 298}
explained variance: [0.564 0.184 0.127 0.051] | PC1+PC2 = 0.748

top loadings:
                PC1   PC2
flat_spell    0.33  0.82
kurtosis      0.58 -0.17
lumpiness     0.50 -0.03
skew          0.47 -0.03
trend_r2     -0.18  0.37
acf1         -0.07  0.21
stability    -0.10  0.18
spec_entropy  0.10 -0.15


In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COL = {"GIFT-Eval": "#8888dd", "TIME": "#66bb88", "BOOM": "#dd8866", "TSBench-Forge": "#222222"}
fig, ax = plt.subplots(figsize=(9, 7))
for bench in ("GIFT-Eval", "TIME", "BOOM", "TSBench-Forge"):
    m = labels == bench
    ax.scatter(P[m, 0], P[m, 1], s=14, alpha=0.55 if bench != "TSBench-Forge" else 0.85,
               c=COL[bench], label=f"{bench} (n={m.sum()})",
               edgecolors="white" if bench == "TSBench-Forge" else "none", linewidths=0.4)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)")
ax.set_title("Structural-feature PCA: TSBench-Forge vs GIFT-Eval / TIME / BOOM")
ax.legend(frameon=False); ax.grid(alpha=0.25)
outdir = SRC.parent / "notebooks" / "figures"; outdir.mkdir(exist_ok=True)
fig.savefig(outdir / "benchmark_pca.png", dpi=150, bbox_inches="tight")
plt.show(); print("saved", outdir / "benchmark_pca.png")

saved /root/TSBench-Forge/notebooks/figures/benchmark_pca.png


## 4. Quantifying spread and overlap

In [9]:
# Spread: total feature-space variance (trace of covariance) per benchmark;
# Overlap: fraction of each benchmark's points whose nearest foreign neighbour is ours.
from scipy.spatial import cKDTree
print(f"{'benchmark':<16} {'spread (trace cov, PC1-4)':>26} {'centroid dist to ours':>22}")
ours = P[labels == "TSBench-Forge"]
for bench in dict.fromkeys(labels):
    m = labels == bench
    spread = float(np.trace(np.cov(P[m].T)))
    cdist = float(np.linalg.norm(P[m].mean(0) - ours.mean(0)))
    print(f"{bench:<16} {spread:>26.2f} {cdist:>22.2f}")

tree = cKDTree(ours[:, :2])
for bench in ("GIFT-Eval", "TIME", "BOOM"):
    m = labels == bench
    d, _ = tree.query(P[m, :2])
    print(f"{bench}: median NN-distance to a TSBench-Forge point = {np.median(d):.2f}")

benchmark         spread (trace cov, PC1-4)  centroid dist to ours
TSBench-Forge                         13.33                   0.00
GIFT-Eval                             10.04                   1.39
TIME                                   6.75                   1.49
BOOM                                  19.21                   1.39
GIFT-Eval: median NN-distance to a TSBench-Forge point = 0.17
TIME: median NN-distance to a TSBench-Forge point = 0.14
BOOM: median NN-distance to a TSBench-Forge point = 0.24


## 5. Reading the results

- **Coverage** — where our black points sit inside an incumbent's cloud, we test the
  same regime on data that post-dates every pretraining cutoff (the anti-memorisation
  claim); regions of an incumbent's cloud we *don't* reach are catalog gaps, and feed
  straight into the source-discovery agent's target list.
- **Spread** — the trace-of-covariance table is the "how much character per sampled
  series" number: higher = broader DGP coverage at equal sample size.
- **BOOM's signature** — the BOOM paper's core claim is that observability series
  (heavy tails, intermittency, flat spells) sit apart from GIFT-Eval; the same axes
  (kurtosis / zero_frac / flat_spell loadings) show whether our web_cloudops slice
  reaches into that region.
- **Caveats** — samples are a few hundred series per benchmark (seeded, re-runnable);
  features are a 12-dim summary, not the full catch22; PCA is linear — treat small
  overlaps/distances as indicative, not exact.